## 활성화 함수 계층 구현하기

### ReLU 계층

- y = x (x > 0)
- y = 0 (x <= 0)

- 순전파 때의 입력인 x가 0보다 크면 역전파는 상류의 값을 그대로 하류로 흘림

- 반면, 순전파 때 x가 0 이하면 역전파 때는 하류로 신호를 보내지 않음 (0을 보넴)

- ReLU 계층 구현

- 신경망 계층의 forward()와 backward() 함수는 넘파이 배열을 인수로 받는다고 가정

In [1]:
class ReLU:
    def __init__(self):
        self.mask = None
    
    def forward(self, x):
        self.mask = (x <= 0)
        out = x.copy()
        out[self.mask] = 0

        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout

        return dx

- ReLU 클래스는 mask라는 인스턴스 변수를 가짐

- mask는 True/False로 구성된 넘파이 배열
- 순전파의 입력인 x의 원소 값이 0 이하인 인덱스는 True, 그 외(0보다 큰 원소)는 False로 유지
- mask 변수는 True/False로 구성된 넘파이 배열을 유지

In [3]:
import numpy as np
x = np.array([[1.0, -0.5], [-2.0, 3.0]])
print(x)

[[ 1.  -0.5]
 [-2.   3. ]]


In [4]:
mask = (x <= 0)
print(mask)

[[False  True]
 [ True False]]


- 순전파 때의 입력 값이 0 이하면 역전파 때의 값은 0

- 그래서 역전파 때는 순전파 때 만들어둔 mask를 써서 mask의 원소가 True인 곳에는 상류에서 전파된 dout을 0으로 설정

### sigmoid 계층

- sigmoid 함수의 계산은 국소적 계산의 전파로 이뤄짐

1. 1단계

- 역전파 때는 상류에서 흘러온 값에 -y^2(순전파의 출력을 제곱한 후 마이너스를 붙인 값)을 곱해서 하류로 전달

2. 2단계

- '+' 노드는 상류의 값을 여과 없이 하류로 보냄

3. 3단계

- exp 노드는 y = exp(x) 연산을 수행

- 상류의 값에 순전파 때의 출력을 곱해 하류로 전파

4. 4단계

- 'x'노드는 순전파 때의 값을 서로 바꿔 곱함

- 간소화 버전은 역전파 과정의 중간 계산들을 생략할 수 있어 더 효율적인 계산임

- 노드를 그룹화하여 sigmoid 계층의 세세한 내용을 노출하지 않고 입력과 출력에만 집중할 수 있음

- sigmoid 계층의 역전파는 순전파의 출력(y)만으로 계산 가능

In [5]:
class Sigmoid():
    def __init__(self):
        self.out = None

    def forward(self, x):
        out = 1 / (1 + np.exp(-x))
        self.out = out
    
        return out
    
    def backward(self, dout):
        dx = dout * (1.0 - self.out) * self.out

        return dx

## Affine/Softmax 계층 구현하기

### Affine 계층


- 신경망의 순전파에서는 가중치 신호의 총합을 계산하기 때문에 행렬의 곱을 사용

In [6]:
X = np.random.rand(2) # 입력
W = np.random.rand(2, 3) # 가중치
B = np.random.rand(3) # 편향

In [7]:
X.shape # (2,)
W.shape # (2, 3)
B.shape #(3,)

(3,)

In [8]:
Y = np.dot(X, W) + B

- X, W, B는 각각 형상이 (2,), (2, 3), (3,)인 다차원 배열

- 뉴런의 가중치 합은 Y = np.dot(X, W) + B처럼 계산
- 이 Y를 활성화 함수로 변환해 다음 층으로 전파하는 것이 신경망 순전파의 흐름


- 행렬의 곱 계산은 대응하는 차원의 원소 수를 일치시키는 게 핵심

- 신경망의 순전파 때 수행하는 행렬의 곱은 기하학에서는 어파인 변환(affine transformation)이라고 함

### 배치용 Affine 계층

- 지금까지 설명한 Affine 계층은 입력 데이터로 X 하나만을 고려한 것

- 배치용 Affine 계층 -> 데이터 N개를 묶어 순전파하는 경우

In [9]:
X_dot_W = np.array([[0, 0, 0], [10, 10, 10]])
B = np.array([1, 2, 3])

In [10]:
X_dot_W

array([[ 0,  0,  0],
       [10, 10, 10]])

In [11]:
X_dot_W + B

array([[ 1,  2,  3],
       [11, 12, 13]])

- 순전파의 편향 덧셈은 각각의 데이터에 더해짐

- 역전파 때는 각 데이터의 역전파 값이 편향의 원소에 모여야 함

In [12]:
dY = np.array([[1, 2, 3], [4, 5, 6]])
dY

array([[1, 2, 3],
       [4, 5, 6]])

In [13]:
dB = np.sum(dY, axis = 0)
dB

array([5, 7, 9])

- 데이터가 2개라고 가정

- 편향의 역전파는 그 두 데이터에 대한 미분을 데이터마다 더해서 구함
- 그래서 np.sum()에서 0번째 축(데이터를 단위로 한 축)에 대해서 (axis = 0)의 총합을 구하는 것

In [14]:
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b

        return out
    
    def backward(self, dout):
        dx = np.dout(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis = 0)

        return dx

### softmax-with-Loss 계층

- 소프트맥스 함수는 입력 값을 정규화하여 출력

- softmax 계층은 입력 값을 정규화(출력의 합이 1이 되도록 변형)하여 출력

- 또한, 숫자는 가짓수가 10개(10클래스 분류)이므로 softmax 계층의 입력은 10개가 됨

- 신경망에서 수행하는 작업은 '학습'과 '추론' 두 가 지

- '추론'할 때는 일반적으로 Softmax 계층을 사용하지 않음
- 또한, 신경망에서 정규화하지 않는 출력 결과에서는 softmax 앞의 Affine 계층의 출력을 점수(score)라 함
- 즉, 신경망의 추론에서 답을 하나만 내는 경우에는 가장 높은 점수만 알면 됨
- - Softmax 계층은 필요 없다
- 반면, 신경망을 학습할 때는 softmax 계층이 필요

- 소프트맥스 계층 구현

- 순실 함수인 교차 엔트로피도 포함 -> Softmax-with-Loss라는 이름으로 구현

- 계산 그래프에서 소프트맥스 함수는 'softmax' 계층으로
- 교차 엔트로피 오차는 'Cross Entropy Error' 계층으로 표기

- 여기에는 3클래스 분류를 가정하고 이전 계층에서 3개의 입력(점수)를 받음
- Softmax 계층은 입력 (a1, a2, a3)을 정규화하여 (y1, y2, y3)을 출력
- Cross Entropy Error 계층은 Softmax의 출력(y1, y2, y3)와 정답 레이블 (t1, t2, t3)를 받고, 이 데이터들로부터 손실 L을 출력

- 역전파의 결과

- Softmax 계층의 역전파는 (y1 - t1, y2 - t2, y3 - t3)라는 결과
- (y1, y2, y3)은 softmax 계층의 출력
- (t1, t2, t3)는 정답 레이블
- (y1 - t1, y2 - t2, y3 - t3)는 softmax 계층의 출력과 정답 레이블의 차분
- 신경망의 역전파에서는 이 차이인 오파가 앞 계층에 전해짐


- 신경망의 목적 -> 신경망의 출력(softmax의 출력)이 정답 레이블과 가까워지도록 가중치 매개변수의 값을 조정하는 것

- 신경망의 출력과 정답 레이블의 오차를 효율적으로 앞 계층에 전달해야 함
- (y1 - t1, y2 - t2, y3 - t3)라는 결과는 바로 softmax 계층의 출려과 정답 레이블의 차이로, 신경망의 현재 출력과 정답 레이블의 오차를 있는 그대로 드러내는 것

In [15]:
# softmax-with-loss 구현
class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None # 손실
        self.y = None # softmax의 출력
        self.t = None # 정답 레이블(원-핫 벡터)

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss
    
    def backward(self, dout = 1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size

        return dx

## 오차역전파법 구하기

### 신경망 학습의 전체 그림

- 전제

- 신경망에는 적응 가능한 가중치와 편향이 있고, 이 가중치와 편향을 훈련 데이터에 적응하도록 조정하는 과정을 '학습'이라 함

1. 1단계 - 미니배치

- 훈련 데이터 중 일부를 무작위로 가져옴

- 이렇게 선별한 데이터를 미니배치라 하며, 그 미니배치의 손실 함수 값을 줄이는 것이 목표

2. 2단계 - 기울기 산출

- 미니배치의 손실 함수 값을 줄이기 위해 각 가중치 매개변수의 기울기를 구함

- 기울기는 손실 함수의 값을 가장 작게 하는 방향을 제시

3. 3단계 - 매개변수 갱신

- 가중치 매개변수를 기울기 방향으로 아주 조금 갱신

4. 4단계 - 반복

- 1 ~ 3단계를 반복

- 지금까지 설명한 오차역전파법이 등장하는 단계는 2단계. 기울기 산출

- 앞에서는 기울기를 구하기 위해서 수치 미분을 사용
- 수치미분은 구현하기는 쉽지만 계산이 오래 걸림
- 오차역전파법을 이용하면 느린 수치 미분과 달리 기울기를 효율적이고 빠르게 구할 수 있음

### 오차역전파법을 적용한 신경망 구현하기

- 2층 신경망을 TwoLayerNet 클래스로 구현

In [ ]:
import sys, os
sys.path.append(os.pardir)  
import numpy as np
from common.layers import *
from common.gradient import numerical_gradient
from collections import OrderedDict


class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std = 0.01):
        # 가중치 초기화
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size) 
        self.params['b2'] = np.zeros(output_size)

        # 계층 생성
        self.layers = OrderedDict()
        self.layers['Affine1'] = Affine(self.params['W1'], self.params['b1'])
        self.layers['Relu1'] = Relu()
        self.layers['Affine2'] = Affine(self.params['W2'], self.params['b2'])

        self.lastLayer = SoftmaxWithLoss()
        
    def predict(self, x):
        for layer in self.layers.values():
            x = layer.forward(x)
        
        return x
        
    # x: 입력 데이터, t: 정답 레이블
    def loss(self, x, t):
        y = self.predict(x)
        return self.lastLayer.forward(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        y = np.argmax(y, axis=1)
        if t.ndim != 1 : t = np.argmax(t, axis=1)
        
        accuracy = np.sum(y == t) / float(x.shape[0])
        return accuracy
        
    # x: 입력 데이터, t: 정답 레이블
    def numerical_gradient(self, x, t):
        loss_W = lambda W: self.loss(x, t)
        
        grads = {}
        grads['W1'] = numerical_gradient(loss_W, self.params['W1'])
        grads['b1'] = numerical_gradient(loss_W, self.params['b1'])
        grads['W2'] = numerical_gradient(loss_W, self.params['W2'])
        grads['b2'] = numerical_gradient(loss_W, self.params['b2'])
        
        return grads
        
    def gradient(self, x, t):
        # 순전파
        self.loss(x, t)

        # 역전파
        dout = 1
        dout = self.lastLayer.backward(dout)
        
        layers = list(self.layers.values())
        layers.reverse()
        for layer in layers:
            dout = layer.backward(dout)

        # 결과 저장
        grads = {}
        grads['W1'], grads['b1'] = self.layers['Affine1'].dW, self.layers['Affine1'].db
        grads['W2'], grads['b2'] = self.layers['Affine2'].dW, self.layers['Affine2'].db

        return grads

- orderedDict은 순서가 있는 딕셔너리
- - 딕셔너리에 추가한 순서를 기억함

- 순전파 때는 추가한 순서대로 각 계층의 forward() 메서드를 호출하기만 하면 처리 완료
- 역전파 때는 계층을 반대 순서로 호출하기만 하면됨

- Affine 계층과 ReLU 계층이 각자의 내부에서 순전파와 역전파를 제대로 처리하고 있으니, 여기에서는 그냥 계층을 올바른 순서대로 연결한 다음 순서대로(혹은 역순으로) 호출하면 끝

### 오차역전파법으로 구한 기울기 검증

- 기울기 구하는 방법

- 1. 수치 미분 사용
- 2. 해석적으로 수식 풀어 구하기
- - 오차역전파법을 이용하여 매개변수가 많아도 효율적으로 계산할 수 있음

- 수치 미분은 느림

- 그리고 오차역전파법을 제대로구현해두면 수치 미분은 오차역전파법을 정확히 구현했는지 확인하기 위해서 필요

- 수치 미분은 구현하기 쉬움
- 그래서 수치 미분의 구현에는 버그가 숨어 있기 어려움

- 오차역전파법은 구현하기 복잡함
- 수치 미분의 결과와 오차역전파법의 결과를 비교하여 오차역전파법으로 제대로 구현했는지 검증

- 두 방식으로 구한 기울기가 일치함(거의 같음)을 확인하는 방법을 기울기 확인(gradient check)이라고 함

In [ ]:
# coding: utf-8
import sys, os
sys.path.append(os.pardir) 
import numpy as np
from dataset.mnist import load_mnist
from two_layer_net import TwoLayerNet

# 데이터 읽기
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

x_batch = x_train[:3]
t_batch = t_train[:3]

grad_numerical = network.numerical_gradient(x_batch, t_batch)
grad_backprop = network.gradient(x_batch, t_batch)

# 각 가중치의 차이의 절댓값을 구한 후, 그 절댓값들의 평균
for key in grad_numerical.keys():
    diff = np.average( np.abs(grad_backprop[key] - grad_numerical[key]) )
    print(key + ":" + str(diff))

- 가장 먼저 MNIST 데이터셋을 읽음

- 그리고 훈련 데이터 일부를 수치 미분으로 구한 기울기와 오차역전파법으로 구한 기울기의 오차를 확인
- - 여기에서는 각 가중치의 매개변수 차이의 절댓값을 구하고, 이를 평균한 값이 오차가 됨

- 수치 미분과 오차역전파법의 결과 오차가 0이 되는 일은 드물다

- 이는 컴퓨터가 할 수 있는 계산의 정밀도가 유한하기 때문
- 이 정말도의 한계 때문에 오차는 대부분이 0이 되지는 않지만, 올바르게 구현했다면 0에 아주 가까운 작은 값이 됨
- 만약 그 값이 크면 오차역전파법을 잘못 구현한 것

### 오차역전파법을 사용한 학습 구현하기

In [ ]:
# coding: utf-8
import sys, os
sys.path.append(os.pardir)

import numpy as np
from dataset.mnist import load_mnist
from two_layer_net import TwoLayerNet

# 데이터 읽기
(x_train, t_train), (x_test, t_test) = load_mnist(normalize=True, one_hot_label=True)

network = TwoLayerNet(input_size=784, hidden_size=50, output_size=10)

iters_num = 10000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.1

train_loss_list = []
train_acc_list = []
test_acc_list = []

iter_per_epoch = max(train_size / batch_size, 1)

for i in range(iters_num):
    batch_mask = np.random.choice(train_size, batch_size)
    x_batch = x_train[batch_mask]
    t_batch = t_train[batch_mask]
    
    # 오차역전파법으로 기울기를 구한다
    grad = network.gradient(x_batch, t_batch)
    
    # 갱신
    for key in ('W1', 'b1', 'W2', 'b2'):
        network.params[key] -= learning_rate * grad[key]
    
    loss = network.loss(x_batch, t_batch)
    train_loss_list.append(loss)
    
    if i % iter_per_epoch == 0:
        train_acc = network.accuracy(x_train, t_train)
        test_acc = network.accuracy(x_test, t_test)
        train_acc_list.append(train_acc)
        test_acc_list.append(test_acc)
        print(train_acc, test_acc)

## 정리


- 계산 그래프를 이용하면 계산 과정을 시각적으로 파악할 수 있음

- 계산 그래프의 노드는 국소적 계산으로 구성됨
- -> 국소적 계산을 조함해 전체 계산을 구성
- 계산 그래프의 순전파는 통상의 계산을 수행
- 계산 그래프의 역전파는 각 노드의 미분을 구할 수 있음

- 신경망의 구성 요소를 계층으로 구현하여 기울기를 효율적으로 계산할 수 있음(오차역전파법)
- 수치 미분과 오차역전파법의 결과를 비교하면 오차역전파법의 구현에 잘목이 없는지 확인할 수 있음(기울기 확인)